# 第5回：モデル評価・ハイパーパラメータ調整 〜 予測の「真価」を問う 〜

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session05_model_evaluation.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

---

## 📋 本日の学習ロードマップ (計90分)

1. **【講義1】正解率の罠：不均衡データの恐怖 (10分)**
   - 1万人に1人の病気をどう見つけるか？ 偽りの 99.9% 精度。
2. **【講義2】混合行列：真の評価指標 5 選 (15分)**
   - 感度・特異度・適合率・F1スコア・陽性的中率。医学における優先順位。
3. **【実習1】混合行列と不均衡データの分析 (15分)**
   - 稀なイベントを検出する難しさを体感する。
4. **【講義3】ROC 曲線と AUC の幾何学的解釈 (15分)**
   - 閾値を動かす。ランダム予測 (0.5) から 完璧な予測 (1.0) へ。
5. **【実習2】ROC 曲線の描画と AUC 算出 (10分)**
   - 2 つの診断ツールの優劣を AUC で裁定する。
6. **【講義4】精度の壁を越える：パラメータ最適化 (15分)**
   - グリッドサーチとランダムサーチ。モデルの設定を極限まで詰める。
7. **【実習3】自動チューニングの実装 (10分)**

---

## 1. セットアップ

精緻な評価グラフを描画するための専門モジュールを使用します。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc, 
    precision_recall_curve, f1_score, balanced_accuracy_score
)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style='whitegrid', palette='mako')

print("モデル監査エンジン、準備完了。")

## 2. 【講義1+2】正解率 (Accuracy) の敗北：不均衡との戦い

### 2.1 健康診断の例
病気の人 ($P$) が 10 人、健康な人 ($N$) が 990 人いる集団を考えます。
- **ダメAI**: 「全員、健康です！」と予測。
- **結果**: $TN = 990, FN = 10$。正解率 = $990/1000 = 99\%$。
- **真実**: この AI はたった 10 人の患者を一人も見つけていません。**正解率 99% なのに医学的にはゴミ**です。

### 2.2 混合行列の 4 指標：$TP, TN, FP, FN$
1. **感度 (Recall)**: 病気の人のうち、何割をすくい上げられたか？ (医学のスクリーニングで最重要)
2. **特異度 (Specificity)**: 健康な人のうち、何割を正しく健康と言えたか？
3. **適合率 (Precision)**: AI が「陽性」と叫んだ時、本当に病気だった確率は？ (空振りの少なさ)
4. **F1 スコア**: 適合率と感度のバランス（調和平均）。

---

### 【実習1】不均衡データによる AI の「偽装」を体験する

In [ ]:
# 癌データセットを加工して、極端な不均衡データ（陽性がわずか 5%）を指定して作成
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

# 陽性データを大幅にカットして不均衡にする
minority = df[df['target'] == 0].sample(frac=0.1, random_state=42) # 悪性を 1/10 に削減
majority = df[df['target'] == 1]
imbalanced_df = pd.concat([minority, majority])

X = imbalanced_df.drop('target', axis=1)
y = imbalanced_df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# 学習
clf = RandomForestClassifier(n_estimators=10)
clf.fit(X_train, y_train)

# 評価
y_pred = clf.predict(X_test)
print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")
print(f"平均正解率 (不均衡考慮): {balanced_accuracy_score(y_test, y_pred):.4f}")
print("\n--- 判決レポート ---")
print(classification_report(y_test, y_pred))

---

## 3. 【講義3】ROC 曲線と言えば医学論文の定番

### 3.1 閾値 (Threshold) という魔術
AI の出力は「確率（例：0.75）」です。これを 0.5 で病気判定するか、それとも 0.1 という厳しい基準で判定するか。この境目を動かすと、「感度」と「特異度」はシーソーのように入れ替わります。

### 3.2 AUC (Area Under the Curve)
この境目を全範囲動かした時の「モデルの地力」を面積で示したものです。
- **1.0**: 神。どんな閾値設定でも完璧。
- **0.8 〜 0.9**: 極めて優秀な診断ツール。
- **0.5**: 当てずっぽう。コイン投げと同等。

---

### 【実習2】ROC 曲線と AUC の描画

In [ ]:
from sklearn.metrics import RocCurveDisplay

fig, ax = plt.subplots(figsize=(8, 7))
RocCurveDisplay.from_estimator(clf, X_test, y_test, ax=ax, name='Random Forest', color='darkorange')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax.set_title('ROC Curve for Rare Disease Detection')
plt.show()

## 4. 【講義4】精度の限界を突破する：ハイパーパラメータ調整

### グリッドサーチか、ランダムサーチか
モデルには人間が決める設定（木の本数、深さなど）があります。これをコンピュータに自動で探させます。
- **Grid Search**: しらみつぶし。確実だが時間がかかる。
- **Random Search**: 運任せだが、広い範囲を高速に探索できる。現代ではこちらが主流になりつつあります。

---

### 【実習3】RandomizedSearchCV による高速チューニング

In [ ]:
param_dist = {
    'n_estimators': [10, 50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 5, 10],
    'criterion': ['gini', 'entropy']
}

# 試行回数 20 回のランダムサーチを実行
search = RandomizedSearchCV(RandomForestClassifier(), param_dist, n_iter=20, cv=3, scoring='f1', random_state=42)
search.fit(X_train, y_train)

print(f"Best F1 Score: {search.best_score_:.4f}")
print(f"Best Parameters: {search.best_params_}")

---

## ✏️ 本日の最終診断ミッション (Master Audit)

**シナリオ**: 不均衡データ `imbalanced_df` において、あなたは「モデルの偏り」を是正する責任者です。

### 課題 1: 特異度 (Specificity) の算出
`classification_report` には「特異度」が載っていません。混合行列を自力で解釈（$TN, FP$ を抽出）し、特異度を計算してください。

### 課題 2: 適合率 $vs$ 再現率のトレードオフ
閾値を 0.1 〜 0.9 まで 0.1 刻みで変更した際、F1スコアが最も高いポイントはどこか、ループ処理を使って見つけ出してください。

### 課題 3: 診断システムの改善案
不均衡データにおいて、単純なランダムフォレストではなく、クラスの重み付け (`class_weight='balanced'`) を設定すると、性能がどのように改善するか（または悪化するか）を確認し、最終的な診断アルゴリズムを決定してください。

---

In [ ]:
# ここに回答を記述してください



---
**Last updated**: 2026-02-15
**Instructor**: Nakaura-T (DS Department, Kumamoto Univ)